## Vanilla MobileNetV3 (Small and Large) Implementation

**AIM: Build and train an image classifier to detect images from different animal species using a Custom MobileNetV3 (Small and Large) Model in TensorFlow.**

### Objectives

- Data visualisation
- Data preprocessing and image augmentation
- Replicate the MobileNetV3 (Small and Large) architecture for model development.
- Compile and train the model
- Add early stopping callback
- Save and load the model
- Model evaluation.
- Make predictions on new data using the trained model.

### Pre-requisite
- Google collaboratry or Jupyter Notebook
- animal-image-classification-dataset
- TensorFlow2

In [ ]:
# Import basic libraries
import os
import sys
import random
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import tensorflow as tf
import pathlib

In [ ]:
# Set seed for reproducibility

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

In [ ]:
# Check GPU availability
!nvidia-smi

In [ ]:
gpus= tf.config.list_physical_devices()

In [ ]:

gpus

In [ ]:
logical_devices = tf.config.list_logical_devices()
logical_devices

In [ ]:
# Check tenorflow version
print("TensorFlow Version", tf.__version__)

In [ ]:
## Set the base path
base_dir = "../../datasets/dog_vs_cats"
base_dir = pathlib.Path(base_dir)
base_dir

In [ ]:
# Train directory
train_dir = base_dir / "train"
train_dir

In [ ]:
# Validation directory
test_dir = base_dir / "test"
test_dir

In [ ]:
# Validation directory
validation_dir = base_dir / "validation"
validation_dir

In [ ]:
## Set Hyperparameters

IMAGE_HEIGHT, IMAGE_WIDTH = 128, 128
BATCH_SIZE = 32
EPOCHS = 300

In [ ]:
# Load the training dataset

train_dataset = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    image_size=(IMAGE_HEIGHT, IMAGE_WIDTH),
    batch_size=BATCH_SIZE,
    seed=SEED,
)

In [ ]:
# Load the validation dataset

validation_dataset = tf.keras.utils.image_dataset_from_directory(
    validation_dir,
    image_size=(IMAGE_HEIGHT, IMAGE_WIDTH),
    batch_size=BATCH_SIZE,
    seed=SEED,
)

In [ ]:
# Get the class names
class_names = train_dataset.class_names
class_names

In [ ]:
# Get the total number of classes
num_classes = len(class_names)
num_classes

In [ ]:
# Sanity check

for images, labels in train_dataset.take(1):
    fixed_images = images.numpy()
    fixed_labels = labels.numpy()


# Visualisations
# No matter how many times you run this cell, the images won change because of teh above

plt.figure(figsize=(12, 12))
for i in range(16):
    ax = plt.subplot(4, 4, i + 1)
    plt.imshow(fixed_images[i].astype("uint8"))
    plt.title(class_names[fixed_labels[i]])
    plt.axis("off")

In [ ]:
# Performance optimization

### Vanilla MobileNetV3 Implementation

In [ ]:
INPUT_SHAPE = (IMAGE_HEIGHT, IMAGE_WIDTH) + (3, )
INPUT_SHAPE

### Utility Scripts

In [2]:
# TensorFlow related imports
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

### Ativations: Hard Sigmoid and Hard Swish

In [3]:
# Hard Swish Activation
class HardSwish(layers.Layer):
    """
    Hard Swish activation function: x * ReLU(x + 3) / 6
    More efficient approximation of Swish for mobile devices
    """

    def __init__(self, **kwargs):
        super(HardSwish, self).__init__(**kwargs)

    def call(self, inputs):
        return inputs * tf.nn.relu6(inputs + 3.0) / 6.0
    
    def get_config(self):
        return super(HardSwish, self).get_onfig()

In [4]:
# Hard Sigmoid Activation
class HardSigmoid(layers.Layer):
    """
    Hard Sigmoid activation function: ReLU6(x + 3) / 3
    More efficient aproximation of Sigmoid for mobile devices.
    """
    def __init__(self, **kwargs):
        super(HardSigmoid, self).__init__(**kwargs)

    def call(self, inputs):
        return tf.nn.relu6(inputs + 3.0) / 6.0
    
    def get_config(self):
        return super(HardSigmoid, self).get_config()

### Squeeze-and-Excitation (SE) Block

In [5]:
class SqueezeExcitation(layers.Layer):
    """
    Squeeze-and-Excitation block for channel attention

    Args:
        filters: Number of filters in the input
        se_ratio: Squeeze ratio for dimensionality redction (default: 0.25)
    """

    def __init__(self, filters, se_ratio=0.25, **kwargs):
        super(SqueezeExcitation, self).__init__(**kwargs)
        self.filters = filters
        self.se_ratio = se_ratio
        self.reduced_filters = max(1, int(filters * se_ratio))

    def build(self, input_shape):
        # Global Average Pooling
        self.gap = layers.GlobalAveragePooling2D(keepdims=True)

        # FC Layers
        self.fc1 = layers.Conv2D(self.reduced_filters,
                                 kernel_size=1,
                                 padding="same",
                                 use_bias=True,
                                 name="se_Reduced")
        
        self.fc2 = layers.Conv2D(self.filters,
                                 kernel_size=1,
                                 padding="same",
                                 use_bias=True,
                                 name="se_expand")
        
        self.relu = layers.ReLU()
        self.hard_sigmoid = HardSigmoid()

    def call(self, inputs):
        # Squeeze: Global pooling
        x = self.gap(inputs)

        # Excitation: FC -> ReLU -> FC -> Hard-Sigmoid
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        x = self.hard_sigmoid(x)

        # Sale
        return inputs * x
    
    def get_config(self):
        config = super(SqueezeExcitation, self).get_config()
        config.update({
            "filters": self.filters,
            "se_ratio": self.se_ratio
        })

        return config

### Inverted Residual Block (MBConv)

In [ ]:
class InvertedResidualBlock(layers.Layer):
    """
    Inverted Residual Blok with expansion, depthwise convolution, and projetion.
    This is the core building block of MobileNetV3
    """

### Inverted Residual Block (with optional SE)

In [ ]:
from tensorflow.keras.layers import (
    Conv2D, DepthwiseConv2D, Activation, BatchNormalization)

In [ ]:
def inverted_residual_block(inputs, expansion, filters, stride, se=False, activation="relu"):
    in_channels = inputs.shape[-1]
    
    # Expansion
    x = Conv2D(in_channels*expansion, 1, padding="same", use_bias=False)(inputs)
    x = BatchNormalization()(x)
    x = Activation(hard_swish if activation=="hswish" else "relu")(x)

    # Depthwise Convolution
    x = DepthwiseConv2D(3, strides=stride, padding="same", use_bias=False)(x)
    x = BatchNormalization()(x)
    x = Activation(hard_swish if activation=="hswish" else "relu")(x)

    # Squeeze-and-Excitation
    if se:
        x = squeeze_exite_block(x)
    
    # Projection Layer
    x = Conv2D(filters, 1, padding="same", use_bias=False)(inputs)
    x = BatchNormalization()(x)

    # Skip Connection if possible
    if stride == 1 and in_channels == filters:
        x = tf.keras.layers.Add()([inputs, x])

    return x



### MobileNetV3 Backbone (MobileNetV3-Small)

In [ ]:
from tensorflow.keras import models
from tensorflow.keras.layers import GlobalAveragePooling2D, Flatten, Reshape

In [ ]:
def MobileNetV3_Small(input_shape=(224, 224, 3), num_classes=1000):
    inputs = tf.keras.layers.Input(shape=input_shape)

    # Initial Layer
    x = Conv2D(16, 3, strides=2, padding="same", use_bias=False)(inputs)
    x = BatchNormalization()(x)
    x = Activation(hard_swish)(x)

    # Sequence of inverted residual blocks
    x = inverted_residual_block(x, expansion=1, filters=16, stride=2, se=True, activation="relu")
    x = inverted_residual_block(x, expansion=round(72/16), filters=24, stride=2, se=False, activation="relu")
    x = inverted_residual_block(x, expansion=round(88/24), filters=24, stride=1, se=False, activation="relu")
    
    x = inverted_residual_block(x, expansion=4, filters=40, stride=2, se=True, activation="hswish")
    x = inverted_residual_block(x, expansion=6, filters=40, stride=1, se=True, activation="hswish")
    x = inverted_residual_block(x, expansion=6, filters=48, stride=1, se=True, activation="hswish")
    x = inverted_residual_block(x, expansion=6, filters=96, stride=2, se=True, activation="hswish")
    x = inverted_residual_block(x, expansion=6, filters=96, stride=1, se=True, activation="hswish")

    # Final Layers
    x = Conv2D(576, 1, use_bias=False)(x)
    x = BatchNormalization()(x)
    x = Activation(hard_swish)(x)

    x = GlobalAveragePooling2D()(x)
    x = Reshape((1, 1, 576))(x)
    x = Conv2D(1024, 1, activation=hard_swish)(x)
    x = Conv2D(num_classes, 1)(x) 
    x = Flatten()(x) 
    outputs = Activation('softmax')(x)

    return models.Model(inputs, outputs)


In [ ]:
model = MobileNetV3_Small(input_shape=INPUT_SHAPE, num_classes=num_classes)
model.summary()

In [ ]:
round(72/16)

In [ ]:
# Compile Model
loss_function = tf.keras.losses.SparseCategoricalCrossentropy()
optimizer=tf.keras.optimizers.Adam(learning_rate=0.000005)
model.compile(
    loss=loss_function,
    optimizer=optimizer,
    metrics=["accuracy"]
)

In [ ]:
# Configure Callbacks

model_checkpoint = tf.keras.callbacks.ModelCheckpoint(
    filepath="models/vanilla_mobilenet_model.keras",
    monitor="val_accuracy",
    save_best_only=True,
    verbose=1
)

early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=10,
    verbose=1,
    restore_best_weights=True
)

reduce_learning_rate = tf.keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss",
    patience=5,
    factor=0.3,
    verbose=1
)

callbacks = [model_checkpoint, early_stopping, reduce_learning_rate]


In [ ]:
# Train the Model to learn patterns from the image

history = model.fit(train_dataset,
                    validation_data=validation_dataset,
                    epochs=EPOCHS,
                    callbacks=callbacks)

In [ ]:
def plot_learning_curves(history):
    acc = history.history["accuracy"]
    val_acc = history.history["val_accuracy"]
    loss = history.history["loss"]
    val_loss = history.history["val_loss"]

    epochs_range = range(len(acc))


    plt.figure(figsize=(18, 7))

    plt.subplot(1, 2, 1)
    plt.plot(epochs_range, acc, label="Training Accuracy")
    plt.plot(epochs_range, val_acc, label="Validation Accuracy")
    plt.legend()
    plt.title("Accuracy")

    plt.subplot(1, 2, 2)
    plt.plot(epochs_range, loss, label="Training Loss")
    plt.plot(epochs_range, val_loss, label="Validation Loss")
    plt.legend()
    plt.title("Loss")

    plt.show()


In [ ]:
loss, accuracy = model.evaluate(validation_dataset)

print(f"Model Loss: {loss:.2f}")
print(f"Model Accuracy: {accuracy:.2f}")